# 02 - The system: from raw files to the follow-up list

This notebook runs the **whole chain from an empty database** with one call and narrates each result:
ingestion with quarantine -> PostgreSQL -> features built in the database -> model -> capacity driven evaluation ->
drift -> price of leakage -> operational list -> prediction path for unseen orders.
Heavy logic lives in the `early_warning` package; cells only call it and show the result.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from IPython.display import display
from early_warning import audit, config, features as F, viz
from early_warning.config import SEED          # the single seed of the whole project
np.random.seed(SEED)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 100)
pd.options.display.float_format = "{:,.4f}".format

import json
from early_warning import demo
from early_warning.db import make_engine
from early_warning.pipeline import run_all
from early_warning.predict import predict
from early_warning.ingestion.loader import SchemaError
ART = config.ARTIFACT_DIR
print("seed:", SEED, "| capacity:", config.CAPACITY, "| boundary:", config.SPLIT_BOUNDARY)

seed: 42 | capacity: 0.1 | boundary: 2018-04-01


## 1. Run the chain
One call: it rebuilds the schemas, so nothing depends on manual state.

In [2]:
results = run_all(external_mode="auto")
engine = make_engine()
manifest = json.loads((ART / "run_manifest.json").read_text())
print("model selected:", results["model"], "| chain time (s):", results["total_seconds"])

model selected: gradient_boosting_deeper | chain time (s): 79.154


## 2. Time budget
Every stage is timed separately. The budget for the whole run is 45 minutes (environment setup is timed by `run_all.py`).

In [3]:
timing = pd.Series(results["stage_seconds"], name="seconds").to_frame()
timing["share_%"] = 100 * timing["seconds"] / timing["seconds"].sum()
display(timing)
print(f"total {results['total_seconds']:.0f} s of {config.TOTAL_BUDGET_MINUTES * 60} s budget")
bootstrap = ART / "bootstrap_timing.json"
if bootstrap.exists():
    display(pd.Series(json.loads(bootstrap.read_text()), name="seconds").to_frame())

,seconds,share_%
database_init,0.1920,0.2426
ingest_validate,9.4750,11.9703
ingest_load_postgres,15.4620,19.5341
ingest_external_api,3.5420,4.4748
features_build_views_and_snapshot,10.9040,13.7757
features_query_benchmark,7.3310,9.2617
model_load_frame_and_split,2.0670,2.6114
model_baseline,0.0230,0.0291
model_train_and_select,6.4540,8.1537
model_evaluate,1.9390,2.4497


total 79 s of 2700 s budget


,seconds
environment_setup,4.6000
pipeline_run_all,84.6000
unit_tests,6.6000
notebooks,144.2000
total_to_final_list_seconds,89.2000
budget_seconds,"2,700.0000"


## 3. Ingestion: contract, quarantine, manifest

In [4]:
tables = pd.DataFrame(manifest["tables"]).T[["rows_in", "rows_clean", "rows_quarantined", "rows_loaded"]]
display(tables)
display(pd.Series(manifest["anomalies"], name="rows").to_frame())
display(pd.DataFrame(manifest["external_sources"]["public_holidays_br"]["years"]).T)
print("notes:", manifest["notes"])

,rows_in,rows_clean,rows_quarantined,rows_loaded
customers,99441,99441,0,99441
geolocation,1000163,1000163,0,1000163
order_items,112650,112650,0,112650
order_payments,103886,103886,0,103886
order_reviews,99224,99224,0,99224
orders,99441,99441,0,99441
product_category_translation,71,71,0,71
products,32951,32951,0,32951
sellers,3095,3095,0,3095


,rows
orders_approved_before_purchase,0
orders_delivered_before_carrier,23
orders_never_approved,160
reviews_answered_before_creation,0


,content_sha256,records,source
2016,8c005b512e2a9951,10,bundled_fallback
2017,5f59af21d3528c56,10,bundled_fallback
2018,dc2395dc5d36d52f,10,bundled_fallback


notes: ['External API unreachable for at least one year: bundled holiday snapshot used.']


The real files are clean, so to show the contract working we corrupt three rows of the reviews file in memory:

In [5]:
display(demo.quarantine_demo(config.DATA_RAW))

,order_id,review_score,review_answer_timestamp,_reason,rows_in,rows_clean
0,73fc7af87114b39712e6da79b0a377eb,7,18-01-2018 21:46,above_max_review_score;,2000,1997
1,a548910a1c6147796b98fdf73dbeba33,5,31-31-2018,invalid_review_answer_timestamp;,2000,1997
2,,5,18-02-2018 14:36,null_order_id;,2000,1997


Rows that break the contract would be written to `data/curated/quarantine/` with a machine-readable reason; the manifest
records counts per reason. Anomalies are legal rows that look suspicious: they are counted, not dropped.
The last table shows where the external signal came from for each year (live API, disk cache or the bundled snapshot).

## 4. Features built in the database, and their temporal audit

In [6]:
display(audit.audit_table()[['column','available_at','available_at_approval','aggregate','source_of_certainty']])

,column,available_at,available_at_approval,aggregate,source_of_certainty
0,purchase_hour,purchase (customer places the order),yes,no,Timestamp is written when the order is placed; approval cannot precede it (E1).
1,purchase_dow,purchase (customer places the order),yes,no,Same as purchase_hour (E1).
2,approval_lag_hours,approval (decision moment),yes,no,Both ends exist at the decision moment by definition (E1). Imputed for never-approved orders.
3,promised_days_from_purchase,purchase (customer places the order),yes,no,The estimate is shown to the customer at checkout; it is never earlier than purchase (E2).
4,promised_days_from_approval,approval (decision moment),yes,no,"Estimate exists since purchase, approval is the decision moment (E2)."
5,ship_limit_slack_days,approval (decision moment),yes,no,Seller deadline is derived from the purchase: it is never earlier than purchase (E3).
6,n_items,purchase (customer places the order),yes,no,The basket is fixed when the order is placed (E4).
7,n_sellers,purchase (customer places the order),yes,no,Basket composition is fixed at purchase (E4).
8,n_products,purchase (customer places the order),yes,no,Basket composition is fixed at purchase (E4).
9,total_price,purchase (customer places the order),yes,no,Prices are shown at checkout (E4).


In [7]:
timing_q = results["feature_query"]
display(pd.Series(timing_q, name="value").to_frame())
plan = (ART / "feature_query_plan.txt").read_text().splitlines()
print("\n".join(plan[:6]), "\n...\n", "\n".join(plan[-2:]))

,value
server_ms_median,252.9000
server_ms_all,"[326.3, 216.8, 211.0, 252.9, 313.5]"
client_fetch_ms_median,"1,094.8000"
budget_ms,500


Hash Left Join  (cost=6953.41..15910.94 rows=99441 width=459) (actual rows=99441 loops=1)
  Hash Cond: (st.order_id = ch.order_id)
  Buffers: shared hit=6208
  ->  Hash Left Join  (cost=3563.98..8531.44 rows=99441 width=371) (actual rows=99441 loops=1)
        Hash Cond: (st.order_id = sh.order_id)
        Buffers: shared hit=5056 
...
 Planning Time: 0.323 ms
Execution Time: 216.257 ms


The serving query (`feat.v_features`) reads pre-computed roll-ups; its server-side execution time is compared to the
500 ms budget above, and the full `EXPLAIN (ANALYZE, BUFFERS)` plan is saved in `data/curated/artifacts/feature_query_plan.txt`.
The expensive part (sorting about 300k event rows per history stream) lives in the refresh procedure, timed in section 2
(`features_build_views_and_snapshot`). The 'client fetch' number includes moving 99k rows into Python and is reported for honesty.

## 5. Split

In [8]:
display(pd.Series(results['split'], name='value').to_frame())

,value
boundary,2018-04-01
embargo_days,46
train_cutoff,2018-02-14
orders_total,99441
excluded_never_approved,160
excluded_no_review,763
train_rows,54753
train_negative_rate,0.1477
embargoed_rows,11050
purged_rows,97


## 6. Baseline rule vs model at 5%, 10% and 20% capacity

In [9]:
cap_model = pd.read_csv(ART / "capacity_table_model.csv")
cap_base = pd.read_csv(ART / "capacity_table_baseline.csv")
print("baseline rule:", results["baseline"]["rule"])
display(cap_base.assign(who="baseline rule"))
display(cap_model.assign(who=results["model"]))
display(pd.read_csv(ART / "model_selection.csv"))

baseline rule: worst_seller_reputation


,capacity,orders_followed_up,negatives_caught,negatives_total,share_of_negatives_caught,precision,lift_vs_random,who
0,5%,1631,377,3753,0.1005,0.2311,2.0089,baseline rule
1,10%,3262,630,3753,0.1679,0.1931,1.6786,baseline rule
2,20%,6524,1075,3753,0.2864,0.1648,1.4321,baseline rule


,capacity,orders_followed_up,negatives_caught,negatives_total,share_of_negatives_caught,precision,lift_vs_random,who
0,5%,1631,586,3753,0.1561,0.3593,3.1226,gradient_boosting_deeper
1,10%,3262,888,3753,0.2366,0.2722,2.3660,gradient_boosting_deeper
2,20%,6524,1392,3753,0.3709,0.2134,1.8544,gradient_boosting_deeper


,model,validation_capture_rate,validation_roc_auc
0,gradient_boosting_deeper,0.2419,0.6466
1,logistic_regression,0.2376,0.6449
2,gradient_boosting_small,0.2360,0.6500


## 7. Operating point: the top 10% of the ranking

In [10]:
op = results["operating_point"]
cm = pd.DataFrame([[op["tp"], op["fn"]], [op["fp"], op["tn"]]],
                  index=["flagged for follow-up", "not flagged"], columns=["negative review", "no negative review"])
display(cm)
display(pd.Series({k: op[k] for k in ["capture_rate", "precision", "recall", "f1", "roc_auc", "pr_auc", "base_rate"]}, name="value").to_frame())
print("95% interval of the capture rate:", [round(x, 4) for x in op["capture_ci95"]])

,negative review,no negative review
flagged for follow-up,888,2865
not flagged,2374,26491


,value
capture_rate,0.2366
precision,0.2722
recall,0.2366
f1,0.2532
roc_auc,0.6376
pr_auc,0.2309
base_rate,0.1151


95% interval of the capture rate: [0.2249, 0.2482]


In [11]:
from early_warning.modeling.dataset import build_modeling_frame, time_split, xy
from early_warning.modeling.train import load_model
from early_warning.modeling.models import score
frame = build_modeling_frame(engine)
split = time_split(frame)
model = load_model()
y_test = split.test["is_negative"].astype(int).to_numpy()
s_model = score(model.pipeline, xy(split.test)[0])
from early_warning.modeling import baseline as B
s_rule = B.rule_scores(split.test, results["baseline"]["rule"])
viz.capacity_curves({"model": (y_test, s_model), "rule baseline": (y_test, s_rule)})

<Figure size 650x400 with 1 Axes>

## 8. Spread over the months of the evaluation period

In [12]:
monthly = pd.read_csv(ART / "monthly_capture_model.csv")
monthly_base = pd.read_csv(ART / "monthly_capture_baseline.csv")
display(monthly)
print("last month:", results["last_month"])
viz.monthly_bars(monthly, monthly_base)

,month,orders,negative_rate,capture_rate,precision,roc_auc
0,2018-04,6726,0.1302,0.2432,0.3165,0.6553
1,2018-05,7028,0.1197,0.2414,0.2888,0.6599
2,2018-06,6139,0.1075,0.2394,0.2573,0.6547
3,2018-07,6130,0.1095,0.2161,0.2365,0.6105
4,2018-08,6594,0.1068,0.2102,0.2242,0.5962


last month: {'month': '2018-08', 'capture_rate': 0.21022727272727273, 'orders': 6594}


<Figure size 700x350 with 1 Axes>

The monthly numbers apply the same 10% quota inside each month. The last month is reported on its own as required;
if it is lower than the pooled number, section 9 shows which features moved.

## 9. Drift between the training and the evaluation period

In [13]:
drift = pd.read_csv(ART / "drift_psi.csv")
display(drift)
viz.psi_bars(drift)
display(pd.read_csv(ART / "monthly_negative_rate.csv"))

,feature,psi,severity,median_shift,aggregate_history_feature
0,s_late_rate_known,0.7892,major,0.0271,True
1,ship_limit_slack_days,0.4148,major,-0.9997,False
2,promised_days_from_approval,0.2991,major,-3.0000,False
3,total_freight,0.2932,major,1.8300,False
4,promised_days_from_purchase,0.2911,major,-3.0000,False
5,approval_lag_hours,0.2127,moderate,0.1167,False
6,product_category,0.1634,moderate,NaN,False
7,s_neg_rate_known,0.1576,moderate,0.0156,True
8,s_known_reviews,0.1518,moderate,35.0000,True
9,s_known_deliveries,0.1455,moderate,34.0000,True


,month,negative_rate,orders
0,2016-09,1.0000,1
1,2016-10,0.2825,315
2,2016-12,0.0000,1
3,2017-01,0.1543,752
4,2017-02,0.1605,1751
5,2017-03,0.1426,2664
6,2017-04,0.1543,2353
7,2017-05,0.1280,3655
8,2017-06,0.1273,3222
9,2017-07,0.1221,3938


PSI above 0.25 is a major shift. The history features of sellers and the promised lead time moved the most: the platform
shortened its delivery promises and sellers' punctuality changed. The negative rate itself fell in the evaluation period, so the
ranking quality (capture rate) must be read against a lower base rate.

## 10. What leakage would have bought: the cheat model

In [14]:
cost = pd.read_csv(ART / "leakage_cost.csv")
display(cost[["model", "extra_columns", "capture_rate", "roc_auc", "pr_auc", "capture_rate_gain_vs_honest", "roc_auc_gain_vs_honest"]])
viz.leakage_bars(cost)

,model,extra_columns,capture_rate,roc_auc,pr_auc,capture_rate_gain_vs_honest,roc_auc_gain_vs_honest
0,honest model (audited columns only),0,0.2366,0.6376,0.2309,0.0000,0.0000
1,"honest + post_approval_operations (status, delivery delay, handover)",4,0.4207,0.7546,0.4429,0.1841,0.1171
2,honest + naive_all_time_seller_negative_rate,1,0.2976,0.7178,0.3110,0.0610,0.0803
3,honest + review_comment_length,1,0.4042,0.7921,0.4150,0.1676,0.1545
4,CHEAT model (all leaky columns),6,0.5422,0.8809,0.6067,0.3056,0.2433


<Figure size 700x350 with 1 Axes>

Same learner, same rows, same split; the only difference is which columns the model may see. The gap between the first row
(honest) and the last row (cheat) is what leakage would have faked on this notebook - a model that is useless the moment it
meets a real, just-approved order.

## 11. Does the external signal help?

In [15]:
display(pd.Series(results['external_ablation'], name='value').to_frame())

,value
validation_capture_with,0.2419
validation_capture_without,0.2413
test_capture_with,0.2366
test_capture_without,0.2358
test_diff_ci95,"[-0.005391554240240459, 0.005042388068319453]"


## 12. The operational output: the follow-up list

In [16]:
follow = pd.read_csv(ART / "follow_up_list_evaluation_period.csv")
print("orders scored:", len(follow), "| flagged for follow-up:", int(follow["follow_up"].sum()))
display(follow.head(10))

orders scored: 32805 | flagged for follow-up: 3281


,rank,order_id,decision_ts,risk_score,follow_up
0,1,05c852fbe20c108caf9f479633cbe088,2018-07-13 15:10:00,0.8684,True
1,2,e55fa3385fa77f6be6a0041c5daf2fb6,2018-08-06 23:31:00,0.8679,True
2,3,2c990009034e1fc83904584b26b5fd25,2018-07-18 10:05:00,0.8658,True
3,4,b635cbb9f7c2bc5de438f7429b8490e5,2018-08-05 23:30:00,0.8646,True
4,5,d6f1fd48e3b9fbaa5ed2794838559afb,2018-06-26 00:18:00,0.8636,True
5,6,940018724d5c0c1847f642dc1d8d0ea2,2018-04-19 14:35:00,0.8626,True
6,7,69fd1a4dd8b96b9ab0dc82b95be21efd,2018-08-05 23:30:00,0.8618,True
7,8,335f8f913a1d29120e07b6bb9847d73c,2018-05-14 15:33:00,0.8613,True
8,9,c2ccb6c2af36d8492e83ecf528f2755c,2018-05-21 16:14:00,0.8611,True
9,10,92a195ed387f8e7f9151eae7a7ec13f0,2018-05-17 21:31:00,0.8606,True


## 13. The prediction path for unseen orders
`python -m early_warning predict --input <file or folder> --output <csv>` validates the schema, applies the same SQL feature
preparation in an isolated scratch copy and returns a ranked list. Below: a folder of newly approved orders (last two weeks of
the data, written as plain CSV files), the ranked output, a consistency check against the main pipeline, and a wrong schema.

In [17]:
drop = demo.make_demo_slice(config.DATA_RAW, approved_from="2018-08-15")
ranked = predict(drop, engine)
print("ranked orders:", len(ranked), "| flagged:", int(ranked["follow_up"].sum()))
display(ranked.head(8))
check = ranked.merge(follow[["order_id", "risk_score"]], on="order_id", suffixes=("_predict", "_pipeline"))
print("max |score difference| vs the main pipeline:", float((check.risk_score_predict - check.risk_score_pipeline).abs().max()))

ranked orders: 2551 | flagged: 256


,rank,order_id,risk_score,follow_up,decision_ts,approval_imputed
0,1,02de1e44936e56ddc1621c46d86bf91e,0.8574,True,2018-08-21 12:35:00,False
1,2,cf5c8d9f52807cb2d2f0a0ff54c478da,0.5421,True,2018-08-24 13:24:00,False
2,3,77df84f9195be22a4e9cb72ca9e8b4c2,0.5339,True,2018-08-16 04:44:00,False
3,4,919baca007d9525b6668c18f79a33197,0.5223,True,2018-08-21 12:50:00,False
4,5,0eb943e25686a73425894fcc95e842e3,0.5060,True,2018-08-16 09:10:00,False
5,6,780dfcd5aaf2662f3f6ce2201164394b,0.5012,True,2018-08-24 04:05:00,False
6,7,5c4e6dc9331f62136e1d51c2af608af9,0.4848,True,2018-08-15 16:24:00,False
7,8,ccec592b58e122dc2bfc2305152d9603,0.4791,True,2018-08-25 16:30:00,False


max |score difference| vs the main pipeline: 9.71445146547012e-17


In [18]:
try:
    predict(config.DATA_CURATED / "demo_orders_missing_status.csv", engine)
except SchemaError as exc:
    print("rejected with a clear error ->", exc)

rejected with a clear error -> 'demo_orders_missing_status.csv' does not match the orders schema; missing columns ['order_status']. Expected columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


## 14. Reading guide for the operations manager
See `SUMMARY_FOR_OPERATIONS.md`. Short version: the list is a *prioritisation aid*, it catches roughly twice as many unhappy
customers as picking orders at random, it is not a crystal ball, and it must be watched monthly because the platform changes.